# 6a · Build an MCP server for colony systems

**Core · Live-code · ~25 min**

An MCP **server** exposes tools and resources over a standard protocol. Ours lives in
[`orbital_mcp_server.py`](orbital_mcp_server.py). Let's see how it's built and what it offers.

> Needs the `mcp` package (installed in setup). This beat doesn't need Ollama.

## Turning a function into an MCP tool

With `FastMCP`, a decorator is all it takes:

```python
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("orbital")

@mcp.tool()
def get_sensor(signal: str) -> str:
    """Get the latest reading for a colony sensor."""
    return tools.get_telemetry(signal, when="latest")
```

The function's **docstring** and **type hints** become the tool's description and schema —
that's what a client (or an LLM) reads to know how to call it.

Open `orbital_mcp_server.py` and read the five `@mcp.tool()` functions and the
`@mcp.resource(...)`.

## Inspect what the server exposes

We can load the server object and ask it for its tools (without starting the transport).

In [ ]:
import importlib.util, os
spec = importlib.util.spec_from_file_location("srv", os.path.abspath("orbital_mcp_server.py"))
srv = importlib.util.module_from_spec(spec); spec.loader.exec_module(srv)

tools = await srv.mcp.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description}")

In [ ]:
# TODO: list the resources too
# resources = await srv.mcp.list_resources()
# for r in resources: print(r.uri, "-", r.name)

## Security by construction

Notice the safe design:
- `get_sensor` rejects unknown signal names.
- `raise_alert` only accepts green/amber/red.
- `control_valve` is **simulated** and returns a "needs human confirmation" message.

In **6b** we'll connect a real client and call these over the protocol.